# ND Week 2 — Linear Regression 2: Lasso, Ridge, and Elastic Net
**DX799 Capstone | Cindy Lopez**

**Research question:** Does regularization improve out-of-sample predictions of marketing outcomes across three independently analyzed datasets? We predict customer-level `ConversionRate` in the primary dataset and campaign-level `Conversions` in the other two. The outcomes are not directly comparable in units.

All preprocessing is fitted on training data only; alpha is selected with five-fold cross-validation on the training set. This notebook contains reproducible code, numerical results, limitations, and an AI-use note.

In [1]:
import numpy as np, pandas as pd, os, warnings
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from statsmodels.stats.outliers_influence import variance_inflation_factor
from IPython.display import display

FILES = {
 'Customer conversion': 'digital_marketing_campaign_dataset.csv',
 'Google Ads': 'GoogleAds_DataAnalytics_Sales_Uncleaned.csv',
 'Product performance': 'marketing_and_product_performance.csv'
}
for label, filename in FILES.items():
    if not os.path.isfile(filename):
        raise FileNotFoundError(f'Missing {filename}. Place all three CSVs in the notebook folder or upload them to Colab.')
print('All three CSV files found.')

All three CSV files found.


## 1. EDA and preparation
The datasets differ in structure, missingness, and target definitions. Google Ads currency fields are converted to numeric; missing predictors are imputed within the training pipeline. Variables that leak target information or occur after the conversion event are excluded.

In [2]:
data={name:pd.read_csv(filename) for name,filename in FILES.items()}
for name, df in data.items():
    print('\n',name,'shape:',df.shape,'missing cells:',int(df.isna().sum().sum()),'duplicates:',int(df.duplicated().sum()))
    display(df.head(2))
    display(df.isna().sum().sort_values(ascending=False).head(5).to_frame('missing'))


 Customer conversion shape: (8000, 20) missing cells: 0 duplicates: 0


,CustomerID,Age,Gender,Income,CampaignChannel,CampaignType,AdSpend,ClickThroughRate,ConversionRate,WebsiteVisits,PagesPerVisit,TimeOnSite,SocialShares,EmailOpens,EmailClicks,PreviousPurchases,LoyaltyPoints,AdvertisingPlatform,AdvertisingTool,Conversion
0,8000,56,Female,136912,Social Media,Awareness,6497.870068,0.043919,0.088031,0,2.399017,7.396803,19,6,9,4,688,IsConfid,ToolConfid,1
1,8001,69,Male,41760,Email,Retention,3898.668606,0.155725,0.182725,42,2.917138,5.352549,5,2,7,2,3459,IsConfid,ToolConfid,1


,missing
CustomerID,0
Age,0
Gender,0
Income,0
CampaignChannel,0



 Google Ads shape: (2600, 13) missing cells: 1150 duplicates: 0


,Ad_ID,Campaign_Name,Clicks,Impressions,Cost,Leads,Conversions,Conversion Rate,Sale_Amount,Ad_Date,Location,Device,Keyword
0,A1000,DataAnalyticsCourse,104.0,4498.0,$231.88,14.0,7.0,0.058,$1892,2024-11-16,hyderabad,desktop,learn data analytics
1,A1001,DataAnalyticsCourse,173.0,5107.0,$216.84,10.0,8.0,0.046,$1679,20-11-2024,hyderabad,mobile,data analytics course


,missing
Conversion Rate,626
Sale_Amount,139
Clicks,112
Cost,97
Conversions,74



 Product performance shape: (10000, 17) missing cells: 0 duplicates: 0


,Campaign_ID,Product_ID,Budget,Clicks,Conversions,Revenue_Generated,ROI,Customer_ID,Subscription_Tier,Subscription_Length,Flash_Sale_ID,Discount_Level,Units_Sold,Bundle_ID,Bundle_Price,Customer_Satisfaction_Post_Refund,Common_Keywords
0,CMP_RLSDVN,PROD_HBJFA3,41770.45,4946,73,15520.09,1.94,CUST_1K7G39,Premium,4,FLASH_1VFK5K,43,34,BNDL_29U6W5,433.80,4,Affordable
1,CMP_JHHUE9,PROD_OE8YNJ,29900.93,570,510,30866.17,0.76,CUST_0DWS6F,Premium,4,FLASH_1M6COK,28,97,BNDL_ULV60J,289.29,2,Innovative


,missing
Campaign_ID,0
Product_ID,0
Budget,0
Clicks,0
Conversions,0


In [3]:
ads=data['Google Ads'].copy()
for col in ['Cost','Sale_Amount']:
    ads[col]=pd.to_numeric(ads[col].astype(str).str.replace(r'[$,]','',regex=True),errors='coerce')
for col in ['Clicks','Impressions','Leads','Conversions','Conversion Rate']:
    ads[col]=pd.to_numeric(ads[col],errors='coerce')
# We deliberately exclude Leads, Conversion Rate and Sale_Amount as outcome-adjacent or downstream variables.
config={
 'Customer conversion':{'frame':data['Customer conversion'],'target':'ConversionRate','numeric':['Age','Income','AdSpend','ClickThroughRate','WebsiteVisits','PagesPerVisit','TimeOnSite','SocialShares','EmailOpens','EmailClicks','PreviousPurchases','LoyaltyPoints'],'categorical':['Gender','CampaignChannel','CampaignType'],'poly':['AdSpend','ClickThroughRate','TimeOnSite']},
 'Google Ads':{'frame':ads,'target':'Conversions','numeric':['Clicks','Impressions','Cost'],'categorical':['Campaign_Name','Location','Device'],'poly':['Clicks','Impressions','Cost']},
 'Product performance':{'frame':data['Product performance'],'target':'Conversions','numeric':['Budget','Clicks','Discount_Level','Bundle_Price'],'categorical':['Subscription_Tier','Common_Keywords'],'poly':['Budget','Clicks','Discount_Level']}
}
for name,c in config.items():
    print(name,'target:',c['target'],'range:',c['frame'][c['target']].min(),c['frame'][c['target']].max())

Customer conversion target: ConversionRate range: 0.0100177828330717 0.1999947079999166
Google Ads target: Conversions range: 3.0 10.0
Product performance target: Conversions range: 1 999


## 2. Regularized regression with training-only cross-validation

Compare unregularized linear regression with Lasso (L1), Ridge (L2), and Elastic Net (L1+L2). The regularization strength `alpha` controls coefficient shrinkage: larger values impose stronger penalties. `l1_ratio` determines the Elastic Net mixture (closer to 1 means more L1). We tune `alpha` over a log-spaced grid and `l1_ratio` over 0.25, 0.5, 0.75. Five-fold CV minimizes RMSE on training folds; test data are used only once for final evaluation. Categorical variables are one-hot encoded, and numeric features are standardized to make penalties comparable.

In [4]:
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.base import clone

def make_encoder():
    try: return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError: return OneHotEncoder(handle_unknown='ignore', sparse=False)

cv=KFold(n_splits=5,shuffle=True,random_state=42)
alpha_grid=np.logspace(-3,2,8)
all_rows=[]; fitted={}
for name,c in config.items():
    cols=c['numeric']+c['categorical']; target=c['target']
    work=c['frame'][cols+[target]].copy()
    work[target]=pd.to_numeric(work[target],errors='coerce')
    work=work.dropna(subset=[target])
    xtr,xte,ytr,yte=train_test_split(work[cols],work[target],test_size=.2,random_state=42)
    prep=ColumnTransformer([
        ('numeric',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]),c['numeric']),
        ('categorical',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('encode',make_encoder())]),c['categorical'])
    ])
    models={
        'Linear baseline':(LinearRegression(),None),
        'Lasso':(Lasso(max_iter=15000,random_state=42),{'model__alpha':alpha_grid}),
        'Ridge':(Ridge(),{'model__alpha':alpha_grid}),
        'Elastic Net':(ElasticNet(max_iter=15000,random_state=42),{'model__alpha':alpha_grid,'model__l1_ratio':[.25,.5,.75]})
    }
    for model_name,(est,grid) in models.items():
        pipe=Pipeline([('prep',clone(prep)),('model',est)])
        if grid:
            search=GridSearchCV(pipe,grid,cv=cv,scoring='neg_root_mean_squared_error',n_jobs=-1,refit=True)
            search.fit(xtr,ytr); fit=search.best_estimator_; params=search.best_params_; cv_rmse=-search.best_score_
        else:
            fit=pipe.fit(xtr,ytr); params={}; cv_rmse=np.nan
        pred=fit.predict(xte)
        coefs=getattr(fit.named_steps['model'],'coef_',np.array([]))
        all_rows.append({'dataset':name,'model':model_name,'train_n':len(ytr),'test_n':len(yte),
            'test_R2':r2_score(yte,pred),'test_RMSE':np.sqrt(mean_squared_error(yte,pred)),
            'test_MAE':mean_absolute_error(yte,pred),'CV_RMSE':cv_rmse,
            'best_alpha':params.get('model__alpha',np.nan),'l1_ratio':params.get('model__l1_ratio',np.nan),
            'nonzero_coefficients':int(np.sum(np.abs(coefs)>1e-8)) if len(coefs) else np.nan})
        fitted[(name,model_name)]=fit
results=pd.DataFrame(all_rows)
display(results.round(4))

,dataset,model,train_n,test_n,test_R2,test_RMSE,test_MAE,CV_RMSE,best_alpha,l1_ratio,nonzero_coefficients
0,Customer conversion,Linear baseline,6400,1600,-0.0023,0.0541,0.0469,NaN,NaN,NaN,23
1,Customer conversion,Lasso,6400,1600,-0.0018,0.0541,0.0469,0.0551,0.0052,NaN,0
2,Customer conversion,Ridge,6400,1600,-0.0019,0.0541,0.0469,0.0552,100.0000,NaN,23
3,Customer conversion,Elastic Net,6400,1600,-0.0018,0.0541,0.0469,0.0551,0.0052,0.50,0
4,Google Ads,Linear baseline,2020,506,0.0007,2.2547,1.9564,NaN,NaN,NaN,20
5,Google Ads,Lasso,2020,506,-0.0000,2.2555,1.9623,2.2766,0.1389,NaN,0
6,Google Ads,Ridge,2020,506,0.0016,2.2538,1.9579,2.2821,100.0000,NaN,20
7,Google Ads,Elastic Net,2020,506,0.0014,2.2539,1.9605,2.2766,0.0268,0.75,4
8,Product performance,Linear baseline,8000,2000,-0.0007,291.0550,252.3043,NaN,NaN,NaN,11
9,Product performance,Lasso,8000,2000,-0.0015,291.1769,252.4421,289.0424,19.3070,NaN,0


## 3. Interpretation: model comparisons and coefficient shrinkage
A lower test RMSE indicates smaller typical prediction error in the target's units. R² below zero indicates performance worse than predicting the held-out mean. Lasso can zero out coefficients; Ridge typically shrinks rather than eliminates coefficients; Elastic Net balances the two. The results are descriptive evidence of predictive performance, not proof of a causal marketing effect. Inspect the output rather than assuming a regularized model wins.

In [5]:
summary=results.loc[results.groupby('dataset')['test_RMSE'].idxmin(),['dataset','model','test_R2','test_RMSE','test_MAE','best_alpha','nonzero_coefficients']]
print('Lowest test RMSE within each dataset:')
display(summary.round(4))
print('Lasso / Elastic Net retained coefficients:')
display(results[results['model'].isin(['Lasso','Elastic Net'])][['dataset','model','nonzero_coefficients','best_alpha','l1_ratio']].round(4))

Lowest test RMSE within each dataset:


,dataset,model,test_R2,test_RMSE,test_MAE,best_alpha,nonzero_coefficients
1,Customer conversion,Lasso,-0.0018,0.0541,0.0469,0.0052,0
6,Google Ads,Ridge,0.0016,2.2538,1.9579,100.0000,20
10,Product performance,Ridge,-0.0006,291.0527,252.3021,100.0000,11


Lasso / Elastic Net retained coefficients:


,dataset,model,nonzero_coefficients,best_alpha,l1_ratio
1,Customer conversion,Lasso,0,0.0052,NaN
3,Customer conversion,Elastic Net,0,0.0052,0.50
5,Google Ads,Lasso,0,0.1389,NaN
7,Google Ads,Elastic Net,4,0.0268,0.75
9,Product performance,Lasso,0,19.3070,NaN
11,Product performance,Elastic Net,11,0.7197,0.25


## 4. Overfitting, expected/unexpected results, and limitations
Five-fold CV tunes penalty strength without accessing the held-out test set. The training-only preprocessing pipeline prevents leakage of test-set distributional information. Regularization is expected to help if correlated predictors or many weak signals make unpenalized coefficients unstable; if the outcome is weakly related to available features, even the best tuned model can perform poorly. Note whether the model's CV RMSE and held-out RMSE are consistent, and whether Lasso or Elastic Net removes many features.

**Limitations:** Google Ads missing values are imputed; this does not establish missingness is random. Conversion counts are nonnegative but linear predictions need not be; count-specific alternatives could be explored in future work. Different datasets and outcome scales should not be compared using raw RMSE. These results are not causal and do not establish how changing ad spend would affect conversions.

## References and AI-use disclosure
- El Kharoua, R. (2024). *Predict conversion in digital marketing dataset*. Kaggle. https://www.kaggle.com/datasets/rabieelkharoua/predict-conversion-in-digital-marketing-dataset
- Nayak, G. (2024). *Google Ads sales dataset*. Kaggle. https://www.kaggle.com/datasets/nayakganesh007/google-ads-sales-dataset
- Shah, I. A. (2024). *Marketing and product performance dataset*. Kaggle. https://www.kaggle.com/datasets/imranalishahh/marketing-and-product-performance-dataset
- Scikit-learn developers. *Linear Models*. https://scikit-learn.org/stable/modules/linear_model.html

**AI use:** ChatGPT assisted with drafting and organizing the code and methodological explanations in response to a request to apply Week 2 regularized regression to three marketing datasets. Numeric results are generated by the executable Python cells; review and verify the results and disclosure before submission.